# 03 · Retraining both models from the K562 CRISPR training pairs

Both models are scikit-learn pipelines: `sign(x)·log1p(|x|)` → standardization → L2 logistic regression (lbfgs). The regularization strength C was tuned by chromosome-grouped five-fold cross-validation; here we refit with the published C and compare with the released coefficients, then report the cross-validated AUPRC.

The released models were fitted with scikit-learn 1.2 and its default lbfgs tolerance (1e-4). Newer scikit-learn versions stop at a slightly different point under that tolerance, so we fit to convergence (tol = 1e-8); the converged fit agrees with the released models to < 0.001 in every coefficient.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from cislink import datasets, load_model
from cislink.model import slog

t = datasets.training()
print(len(t), "pairs,", int(t.label.sum()), "positives; folds", sorted(t.fold.unique()))

10250 pairs, 469 positives; folds [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [2]:
def pipe(C):
    return make_pipeline(FunctionTransformer(slog), StandardScaler(), LogisticRegression(C=C, solver="lbfgs", tol=1e-8, max_iter=100000))

res = []
for m in ("e2g", "v2g"):
    model = load_model(m); C = model.meta["C"]
    x = t[list(model.features)].fillna({"HiC_Contact": 0, "HiC_megamap_oe": 0})
    p = pipe(C).fit(x, t.label)
    coef = p.named_steps["logisticregression"].coef_[0]
    oof = np.zeros(len(t))
    for f in sorted(t.fold.unique()):
        tr = (t.fold != f).values
        oof[~tr] = pipe(C).fit(x[tr], t.label[tr]).predict_proba(x[~tr])[:, 1]
    res.append({"model": model.name, "C": round(C, 4), "max |coef - released|": np.abs(coef - model.coef).max(),
                "max |score - released| (training pairs)": np.abs(p.predict_proba(x)[:, 1] - model.predict(t)).max(),
                "5-fold CV AUPRC (chromosome folds)": average_precision_score(t.label, oof)})
pd.DataFrame(res)

,model,C,max |coef - released|,max |score - released| (training pairs),5-fold CV AUPRC (chromosome folds)
0,CisLink-E2G,0.3299,0.000939,0.000268,0.688225
1,CisLink-V2G,0.3347,0.000287,0.000082,0.644979


The converged refit reproduces the released models: coefficients agree to < 0.001 and scores to < 0.001.